In [ ]:

import matplotlib.pyplot as plt
from src.data.fetcher import fetch_historical
from src.portfolio.simulator import Simulator
from src.agents.agents import MomentumAgent
import yfinance as yf

data = yf.download("AAPL", start="2020-01-01", end="2023-01-01", auto_adjust=True)
#print(data.head())
import pandas as pd

# If you saved to CSV
df = pd.read_csv("data/AAPL.csv", header=[0])  # read first row as header

# If multiindex got saved, collapse it
df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

# Rename columns properly
df.columns = ["Date", "Open", "High", "Low", "Close", "Volume"]

print(df.head())


# Fetch data
df = fetch_historical("AAPL", start="2022-01-01", end="2023-01-01")
# flatten MultiIndex columns
df.columns = [col[0] for col in df.columns]

print(df.head())
print(df.columns)


# Run simulation
agent = MomentumAgent(window=5)
sim = Simulator(df, agent, cash=100000)
results = sim.run()

# Plot portfolio value
results.plot(x="Date", y="PortfolioValue", title="Momentum Backtest", figsize=(10,5))
plt.show()


In [ ]:

import yfinance as yf
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def get_stock(symbol, start="2020-01-01", end="2023-01-01"):
    file_path = DATA_DIR / f"{symbol}.csv"

    if file_path.exists():
        print(f"Loading {symbol} from local file...")
        df = pd.read_csv(file_path, parse_dates=["Date"])
    else:
        print(f"Fetching {symbol} from Yahoo Finance...")
        df = yf.download(symbol, start=start, end=end, auto_adjust=True)
        df = df.reset_index()  # make Date a column
        df.to_csv(file_path, index=False)  # save clean CSV

    return df

if __name__ == "__main__":
    df = get_stock("AAPL")
    print(df.head())


In [ ]:
import pandas as pd
from src.metrics import performance

# fake portfolio values for demo
values = pd.Series([100, 105, 95, 120, 110, 130])

print("Cumulative Returns:", performance.cumulative_returns(values))
print("Annualized Returns:", performance.annualized_returns(values, periods_per_year=252))
print("Sharpe Ratio:", performance.sharpe_ratio(values))
print("Max Drawdown:", performance.max_drawdown(values))


In [ ]:
# Example in Backtester
portfolio = PortfolioManager(initial_cash=100_000)
for i, row in df.iterrows():
    price = row["Close"]
    action = agent.decide_action(row, i, df)
    if action == "BUY":
        portfolio.buy("AAPL", price, 10)
    elif action == "SELL":
        portfolio.sell("AAPL", price, 10)
